# Exploración inicial del fichero del INE (notebook de trabajo)

Este es mi notebook de trabajo previo al `main.ipynb`: aquí fue donde descubrí cómo venía el
fichero del INE, qué columnas estaban vacías y qué códigos usaba cada variable. Trabajo sobre
la muestra del repo para que sea rápido de ejecutar.

In [1]:
import pandas as pd
import sys
sys.path.append("../utils")
from utils_proyecto import PROV_A_CCAA, calidad_datos

df = pd.read_parquet("../data_sample/muestra_nacimientos_2024.parquet")
print(df.shape)
df.head()

(25000, 102)


,PROI,MUNI,MESPAR,ANOPAR,PROPAR,MUNPAR,LUGARPA,ASISTIDO,MULTIPLI,NORMA,...,CODCA1N,CODCA2N,CODCA4N,CLASIF,SORDENV,NUMHVT,TMUNPAR,RELAM,RELAP,RELLB2
198673,14,021,12,2024.0,14,021,1,1,1.0,1,...,NaN,NaN,NaN,3,1.0,1.0,6,,,
37801,03,076,12,2024.0,03,076,1,1,1.0,1,...,NaN,NaN,NaN,3,1.0,1.0,2,,,
86339,20,069,07,2024.0,20,069,1,1,1.0,2,...,NaN,NaN,NaN,3,1.0,1.0,6,,,
13569,28,013,09,2024.0,28,013,1,1,1.0,1,...,NaN,NaN,NaN,3,1.0,1.0,4,,,
73931,14,021,04,2024.0,14,021,1,1,1.0,1,...,NaN,NaN,NaN,3,1.0,1.0,6,,,


## ¿Qué columnas están realmente vacías?

El INE usa cadena vacía '' en vez de NaN en muchas categóricas, así que un `isna()` normal
engaña. Miro las dos cosas a la vez:

In [2]:
vacias = {}
for col in df.columns:
    if df[col].dtype == object or str(df[col].dtype).startswith("str"):
        pct = ((df[col].isna()) | (df[col].astype(str).str.strip() == "")).mean() * 100
    else:
        pct = df[col].isna().mean() * 100
    vacias[col] = round(pct, 1)

vacias_s = pd.Series(vacias).sort_values(ascending=False)
print("Columnas con mas del 50% vacio/nulo:")
print(vacias_s[vacias_s > 50].to_string())

Columnas con mas del 50% vacio/nulo:
ESTUDIOM     100.0
CAUTOM       100.0
AUTOPSN      100.0
MUERN        100.0
CAUTOP       100.0
ESTUDIOP     100.0
CODCA4N      100.0
RELLB2       100.0
RELAP        100.0
RELAM        100.0
CODCA1N      100.0
RELLB1       100.0
CODCA2N      100.0
PAISHANTX     96.1
ANOREL        94.5
EDADMREL      94.5
MESEST1       93.0
ANIOEST1      92.9
ESTABLE1      92.0
PHECHO        80.1
PAISNAHA      74.6
NACIOXHA      74.6
NACIOEHA      74.6
CASPNM        69.9
MUNHANTE      61.4
PROHANTE      58.7
ANIOHAN       54.6
MESHAN        54.6
ANIOMAT       51.2
EDADMM        51.2
CUANNACP      51.2
ANOCA         51.2
MESMAT        51.1


Aquí es donde vi que **estudios y ocupación (ESTUDIOM, CAUTOM, RELAM...) vienen 100% vacías**
en el fichero 2024, y que las fechas de matrimonio solo existen para las casadas. Estas
decisiones están justificadas y aplicadas en el `main.ipynb` (Paso 2).

In [3]:
# comprobacion rapida de la tasa de cesarea en la muestra
print("Tasa de cesarea en la muestra:", round((df["CESAREA"] == "1").mean() * 100, 1), "%")

# y por CCAA
ccaa = df["PROPAR"].map(PROV_A_CCAA)
tasa = (df.groupby(ccaa)["CESAREA"].apply(lambda s: (s == "1").mean() * 100)
          .sort_values(ascending=False).round(1))
print()
print(tasa.to_string())

Tasa de cesarea en la muestra: 26.4 %

PROPAR
Melilla               32.4
Extremadura           30.0
C. Valenciana         29.5
Murcia                29.5
Cataluña              28.7
Castilla-La Mancha    28.0
Galicia               27.6
Andalucía             27.2
Baleares              26.6
Castilla y León       25.6
Canarias              25.5
Madrid                24.5
Cantabria             24.1
La Rioja              22.2
Aragón                20.0
Asturias              19.1
País Vasco            17.5
Ceuta                 15.8
Navarra               14.6


Con esto confirmé que el problema tenía señal suficiente y pasé a montar el `main.ipynb`
definitivo con el proceso completo.